# Import Libraries

In [3]:
import pandas as pd
import seaborn as sns
import matplotlib as plt
import numpy as np
import os
from datetime import datetime

# Load data

In [2]:
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "projects", "sales_prediction", "datasets")

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"not found: {DATA_DIR}")

sales_train = pd.read_csv(os.path.join(DATA_DIR, "cleaned_sales.csv"))
items = pd.read_csv(os.path.join(DATA_DIR, "items.csv"))
item_categories = pd.read_csv(os.path.join(DATA_DIR, "item_categories.csv"))
shops = pd.read_csv(os.path.join(DATA_DIR, "shops.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

FileNotFoundError: not found: e:\sales_predict\notebooks\projects\sales_prediction\datasets

# Merging data

In [ ]:
sales_train

,date_block_num,shop_id,item_id,date,item_cnt_day,item_price
0,0,2,27,2013-01-11,1.0,2499.0
1,0,2,33,2013-01-05,1.0,499.0
2,0,2,317,2013-01-04,1.0,299.0
3,0,2,438,2013-01-19,1.0,299.0
4,0,2,471,2013-01-09,1.0,399.0
...,...,...,...,...,...,...
2928021,33,59,22088,2015-10-03,1.0,119.0
2928022,33,59,22088,2015-10-27,1.0,119.0
2928023,33,59,22091,2015-10-03,1.0,179.0
2928024,33,59,22100,2015-10-18,1.0,629.0


In [ ]:
df = sales_train.merge(items, on='item_id', how='left')
df = df.merge(item_categories, on='item_category_id', how='left')
expanded_df = df.merge(shops, on='shop_id', how='left')
expanded_df.head()

,date_block_num,shop_id,item_id,date,item_cnt_day,item_price,item_name,item_category_id,item_category_name,shop_name
0,0,2,27,2013-01-11,1.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,"Адыгея ТЦ ""Мега"""
1,0,2,33,2013-01-05,1.0,499.0,1+1 (BD),37,Кино - Blu-Ray,"Адыгея ТЦ ""Мега"""
2,0,2,317,2013-01-04,1.0,299.0,1С:Аудиокниги. Мединский В. Мифы о России. О р...,45,Книги - Аудиокниги 1С,"Адыгея ТЦ ""Мега"""
3,0,2,438,2013-01-19,1.0,299.0,1С:Аудиотеатр. Лучшие произведения русских пис...,45,Книги - Аудиокниги 1С,"Адыгея ТЦ ""Мега"""
4,0,2,471,2013-01-09,1.0,399.0,1С:Бухгалтерия 8 (ред.3.0) как на ладони. Изд ...,49,Книги - Методические материалы 1С,"Адыгея ТЦ ""Мега"""


# Expand monthly grid

In [ ]:
expanded_df['item_cnt_month'] = df.groupby(['date_block_num', 'shop_id', 'item_id'])['item_cnt_day'].transform('sum')

In [ ]:
all_dates = expanded_df['date_block_num'].unique()
all_shops = expanded_df['shop_id'].unique()
all_items = expanded_df['item_id'].unique()

full_grid = pd.MultiIndex.from_product(
    [all_dates, all_shops, all_items],
    names=['date_block_num', 'shop_id', 'item_id']
).to_frame(index=False)

expanded_df = full_grid.merge(
    expanded_df,
    on=['date_block_num', 'shop_id', 'item_id'],
    how='left'
)

expanded_df['item_cnt_month'] = expanded_df['item_cnt_month'].fillna(0)

expanded_df = expanded_df.sort_values(['shop_id', 'item_id', 'date_block_num'])

In [ ]:
expanded_df

,date_block_num,shop_id,item_id,date,item_cnt_day,item_price,item_name,item_category_id,item_category_name,shop_name,item_cnt_month
17810,0,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1290791,1,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2560348,2,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3838376,3,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
5098685,4,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...
37615390,29,59,22169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
38858317,30,59,22169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
40102230,31,59,22169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
41344407,32,59,22169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [ ]:
expanded_df['item_name'] = expanded_df['item_id'].map(items.set_index('item_id')['item_name'])
expanded_df['item_category_id'] = expanded_df['item_id'].map(items.set_index('item_id')['item_category_id'])
expanded_df['item_category_name'] = expanded_df['item_category_id'].map(item_categories.set_index('item_category_id')['item_category_name'])
expanded_df['shop_name'] = expanded_df['shop_id'].map(shops.set_index('shop_id')['shop_name'])
expanded_df.head()

,date_block_num,shop_id,item_id,date,item_cnt_day,item_price,item_name,item_category_id,item_category_name,shop_name,item_cnt_month
17810,0,2,0,NaN,NaN,NaN,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0
1290791,1,2,0,NaN,NaN,NaN,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0
2560348,2,2,0,NaN,NaN,NaN,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0
3838376,3,2,0,NaN,NaN,NaN,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0
5098685,4,2,0,NaN,NaN,NaN,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0


In [ ]:
print(expanded_df.columns)

Index(['date_block_num', 'shop_id', 'item_id', 'date', 'item_cnt_day',
       'item_price', 'item_name', 'item_category_id', 'item_category_name',
       'shop_name', 'item_cnt_month'],
      dtype='object')


In [ ]:
expanded_df.isna().sum()

date_block_num               0
shop_id                      0
item_id                      0
date                  39907442
item_cnt_day          39907442
item_price            39907442
item_name                    0
item_category_id             0
item_category_name           0
shop_name                    0
item_cnt_month               0
dtype: int64

# Fill Nan

## Fill Nan in item_price as last saled item_price

In [ ]:
expanded_df['item_price'] = (
    expanded_df.sort_values(['item_id', 'date_block_num'])
               .groupby('item_id')['item_price']
               .ffill()
)

In [ ]:
expanded_df.isna().sum()

date_block_num               0
shop_id                      0
item_id                      0
date                  39907442
item_cnt_day          39907442
item_price            11034307
item_name                    0
item_category_id             0
item_category_name           0
shop_name                    0
item_cnt_month               0
dtype: int64

if there are still some Nan for example first time item_sold, we can fill with future price it doesn't affect our model beacuse there are 0 sales for this items

In [ ]:
expanded_df['item_price'] = (
    expanded_df.sort_values(['item_id', 'date_block_num'])
               .groupby('item_id')['item_price']
               .bfill()
)

In [ ]:
expanded_df.isna().sum()

date_block_num               0
shop_id                      0
item_id                      0
date                  39907442
item_cnt_day          39907442
item_price                   0
item_name                    0
item_category_id             0
item_category_name           0
shop_name                    0
item_cnt_month               0
dtype: int64

we can fill item_cnt_day as 0 because there were no sales

In [ ]:
expanded_df['item_cnt_day'] = expanded_df['item_cnt_day'].fillna(0)

but we still have a date that is actually useless since there is already a date_block_num feature and we predict things not by the day, so we can easily throw it away

In [ ]:
expanded_df.drop(columns=['date'], inplace=True)

In [ ]:
expanded_df.duplicated().sum()

np.int64(1048770)

# Feature Engineering

In [ ]:
#dates = pd.to_datetime(expanded_df["date"], format="%d-%m-%Y", errors="coerce")
#mask = dates.isna()
#dates.loc[mask] = pd.to_datetime(expanded_df.loc[mask, "date"], format="%Y-%m-%d", errors="coerce")
#expanded_df["date"] = dates.dt.strftime("%d-%m-%Y")

# Features depending on date

### Year, week, month, day of week, weekend

We give the model an explicit understanding of information about a specific point in time because it will not be able to establish patterns if we simply leave the date attribute

## Season

In [ ]:
def get_season_from_block_num(block_num):
    month = (block_num % 12) + 1
    if month in [12, 1, 2]:
        return "Зима"
    elif month in [3, 4, 5]:
        return "Весна"
    elif month in [6, 7, 8]:
        return "Лето"
    else:
        return "Осень"

expanded_df['season'] = expanded_df['date_block_num'].apply(get_season_from_block_num)

### Cycle features

December is the 11th month and January is the 0th month, in this example it is clearly seen that the model will not understand that they are neighbors, so the months need to be transferred to an imaginary circle, where the sin values ​​will be the x values, and the cosine values ​​will be the y values, respectively.

In [ ]:
expanded_df['month_sin'] = np.sin(2 * np.pi * expanded_df['date_block_num'] / 12)
expanded_df['month_cos'] = np.cos(2 * np.pi * expanded_df['date_block_num'] / 12)

### Lag and rolling statistics features

In [ ]:
expanded_df = expanded_df.sort_values(['shop_id', 'item_id', 'date_block_num'])

# Add lags

In [ ]:
def add_lag_features(df, group_cols, target_col, lags):
    df = df.sort_values(group_cols + ['date_block_num']).copy()
    for lag in lags:
        lag_col = f'{target_col}_lag_{lag}'
        df[lag_col] = (
            df.groupby(group_cols)[target_col]
              .shift(lag)
              .fillna(0)
              .astype(np.float32)
        )
    return df

In [ ]:
expanded_df = add_lag_features(expanded_df, ['shop_id', 'item_id'], 'item_cnt_month', [1, 2, 3, 12])

In [ ]:
expanded_df

,date_block_num,shop_id,item_id,item_cnt_day,item_price,item_name,item_category_id,item_category_name,shop_name,item_cnt_month,season,month_sin,month_cos,item_cnt_month_lag_1,item_cnt_month_lag_2,item_cnt_month_lag_3,item_cnt_month_lag_12
17810,0,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,Зима,0.000000e+00,1.000000e+00,0.0,0.0,0.0,0.0
1290791,1,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,Зима,5.000000e-01,8.660254e-01,0.0,0.0,0.0,0.0
2560348,2,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,Весна,8.660254e-01,5.000000e-01,0.0,0.0,0.0,0.0
3838376,3,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,Весна,1.000000e+00,6.123234e-17,0.0,0.0,0.0,0.0
5098685,4,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,Весна,8.660254e-01,-5.000000e-01,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37615390,29,59,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Ярославль ТЦ ""Альтаир""",0.0,Лето,5.000000e-01,-8.660254e-01,0.0,0.0,0.0,0.0
38858317,30,59,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Ярославль ТЦ ""Альтаир""",0.0,Лето,2.388680e-15,-1.000000e+00,0.0,0.0,0.0,0.0
40102230,31,59,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Ярославль ТЦ ""Альтаир""",0.0,Лето,-5.000000e-01,-8.660254e-01,0.0,0.0,0.0,0.0
41344407,32,59,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Ярославль ТЦ ""Альтаир""",0.0,Осень,-8.660254e-01,-5.000000e-01,0.0,0.0,0.0,0.0


In [ ]:
expanded_df = expanded_df.sort_values(['shop_id', 'item_id', 'date_block_num'])
grouped = expanded_df.groupby(['shop_id', 'item_id', 'date_block_num'])

expanded_df['rolling_median_6'] = (
    grouped['item_cnt_month']
    .shift(1)
    .rolling(window=6, min_periods=1)
    .median()
    .transform(lambda x: x)
)


expanded_df['rolling_median_3'] = (
    grouped['item_cnt_month']
    .shift(1)
    .rolling(window=3, min_periods=1)
    .median()
    .transform(lambda x: x)
)

cols_to_fill = [
    'rolling_median_6',
    'rolling_median_3'
]
expanded_df[cols_to_fill] = expanded_df[cols_to_fill].fillna(0)

## Data checkpoint

In [ ]:
expanded_df.to_pickle("checkpoint.pkl")

In [ ]:
expanded_df = pd.read_pickle("checkpoint.pkl")

## Groupping

In [ ]:
expanded_df.columns

Index(['date_block_num', 'shop_id', 'item_id', 'item_cnt_day', 'item_price',
       'item_name', 'item_category_id', 'item_category_name', 'shop_name',
       'item_cnt_month', 'season', 'month_sin', 'month_cos',
       'item_cnt_month_lag_1', 'item_cnt_month_lag_2', 'item_cnt_month_lag_3',
       'item_cnt_month_lag_12', 'rolling_median_6', 'rolling_median_3'],
      dtype='object')

In [ ]:
expanded_df.isna().sum()

date_block_num           0
shop_id                  0
item_id                  0
item_cnt_day             0
item_price               0
item_name                0
item_category_id         0
item_category_name       0
shop_name                0
item_cnt_month           0
season                   0
month_sin                0
month_cos                0
item_cnt_month_lag_1     0
item_cnt_month_lag_2     0
item_cnt_month_lag_3     0
item_cnt_month_lag_12    0
rolling_median_6         0
rolling_median_3         0
dtype: int64

In [ ]:
expanded_df = expanded_df.sort_values(['shop_id', 'item_id', 'date_block_num'])

grouped = expanded_df.groupby(['shop_id', 'item_id'])

expanded_df['item_avg_sales_month_lag1'] = (
    expanded_df.groupby(['item_id', 'date_block_num'])['item_cnt_month_lag_1'].transform('median')
)

expanded_df['shop_item_avg_lag1'] = (
    expanded_df.groupby(['shop_id', 'item_id'])['item_cnt_month_lag_1'].transform('median')
)

expanded_df[[ 'item_avg_sales_month_lag1', 'shop_item_avg_lag1']] = (
    expanded_df[['item_avg_sales_month_lag1', 'shop_item_avg_lag1']].fillna(0)
)

## Logarifmic feature

why do we need a logarithm at all?

It's simple, on some days sales skyrocketed due to the release of popular games, for example, a logarithm can partially solve this problem

In [ ]:
expanded_df['log_item_cnt_month_lag_1'] = np.log1p(expanded_df['item_cnt_month_lag_1'])

In [ ]:
expanded_df['log_item_cnt_month_lag_1'].isna().sum()

np.int64(0)

In [ ]:
mask = expanded_df['log_item_cnt_month_lag_1'].isna()
problem_rows = expanded_df.loc[mask, 'item_cnt_month_lag_1']
print(problem_rows.value_counts())

Series([], Name: count, dtype: int64)


In [ ]:
expanded_df

,date_block_num,shop_id,item_id,item_cnt_day,item_price,item_name,item_category_id,item_category_name,shop_name,item_cnt_month,...,month_cos,item_cnt_month_lag_1,item_cnt_month_lag_2,item_cnt_month_lag_3,item_cnt_month_lag_12,rolling_median_6,rolling_median_3,item_avg_sales_month_lag1,shop_item_avg_lag1,log_item_cnt_month_lag_1
17810,0,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,...,1.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1290791,1,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,...,8.660254e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2560348,2,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,...,5.000000e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3838376,3,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,...,6.123234e-17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5098685,4,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,...,-5.000000e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37615390,29,59,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Ярославль ТЦ ""Альтаир""",0.0,...,-8.660254e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38858317,30,59,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Ярославль ТЦ ""Альтаир""",0.0,...,-1.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
40102230,31,59,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Ярославль ТЦ ""Альтаир""",0.0,...,-8.660254e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41344407,32,59,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Ярославль ТЦ ""Альтаир""",0.0,...,-5.000000e-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## City

In [ ]:
expanded_df['city'] = expanded_df['shop_name'].str.split().str[0]

In [ ]:
expanded_df['city'].unique()

array(['Адыгея', 'Балашиха', 'Волжский', 'Вологда', 'Воронеж', 'Выездная',
       'Жуковский', 'Интернет-магазин', 'Казань', 'Калуга', 'Коломна',
       'Красноярск', 'Курск', 'Москва', 'Мытищи', 'Н.Новгород',
       'Новосибирск', 'Омск', 'РостовНаДону', 'СПб', 'Самара', 'Сергиев',
       'Сургут', 'Томск', 'Тюмень', 'Уфа', 'Химки', 'Цифровой', 'Чехов',
       'Якутск', 'Ярославль'], dtype=object)

may be confusing digital and online store, but since these are separate categories and logically different places, then everything is fine

## Type of shop

In [ ]:
expanded_df['type_of_shop'] = expanded_df['shop_name'].str.split().str[1]

In [ ]:
expanded_df['type_of_shop'].unique()

array(['ТЦ', 'ТРК', 'ТРЦ', '(Плехановская,', 'Торговля', 'ул.', 'ЧС',
       '"Распродажа"', 'МТРЦ', 'Магазин', 'ТК', 'Посад', 'склад',
       'Орджоникидзе,'], dtype=object)

## Price changes

In [ ]:
expanded_df.sort_values(['item_id', 'date_block_num'], inplace=True)
expanded_df.loc[:, 'price_change'] = expanded_df.groupby('item_id')['item_price'].diff()
expanded_df.loc[:, 'price_increasing'] = (expanded_df['price_change'] > 0).astype(int)
expanded_df.groupby('item_id')['price_increasing'].transform(lambda x: x.fillna(0))
expanded_df.drop(columns=['price_change'], inplace=True)

In [ ]:
expanded_df['item_min_price'] = expanded_df.groupby('item_id')['item_price'].transform('min')
expanded_df['item_max_price'] = expanded_df.groupby('item_id')['item_price'].transform('max')

In [ ]:
expanded_df['price_range'] = expanded_df['item_max_price'] - expanded_df['item_min_price']

In [ ]:
expanded_df.columns

Index(['date_block_num', 'shop_id', 'item_id', 'item_cnt_day', 'item_price',
       'item_name', 'item_category_id', 'item_category_name', 'shop_name',
       'item_cnt_month', 'season', 'month_sin', 'month_cos',
       'item_cnt_month_lag_1', 'item_cnt_month_lag_2', 'item_cnt_month_lag_3',
       'item_cnt_month_lag_12', 'rolling_median_6', 'rolling_median_3',
       'item_avg_sales_month_lag1', 'shop_item_avg_lag1',
       'log_item_cnt_month_lag_1', 'city', 'type_of_shop', 'price_increasing',
       'item_min_price', 'item_max_price', 'price_range'],
      dtype='object')

# Downcasting

In [ ]:
def downcast_dtypes(df):
    float_cols = df.select_dtypes(include=['float64']).columns
    int_cols = df.select_dtypes(include=['int64']).columns

    df[float_cols] = df[float_cols].astype('float32')
    df[int_cols] = df[int_cols].astype('int32')

    return df

In [ ]:
downcast_dtypes(expanded_df)

,date_block_num,shop_id,item_id,item_cnt_day,item_price,item_name,item_category_id,item_category_name,shop_name,item_cnt_month,...,rolling_median_3,item_avg_sales_month_lag1,shop_item_avg_lag1,log_item_cnt_month_lag_1,city,type_of_shop,price_increasing,item_min_price,item_max_price,price_range
17810,0,2,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Адыгея ТЦ ""Мега""",0.0,...,0.0,0.0,0.0,0.0,Адыгея,ТЦ,0,58.0,58.0,0.0
39819,0,3,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Балашиха ТРК ""Октябрь-Киномир""",0.0,...,0.0,0.0,0.0,0.0,Балашиха,ТРК,0,58.0,58.0,0.0
62453,0,4,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Волжский ТЦ ""Волга Молл""",0.0,...,0.0,0.0,0.0,0.0,Волжский,ТЦ,0,58.0,58.0,0.0
1050817,0,5,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Вологда ТРЦ ""Мармелад""",0.0,...,0.0,0.0,0.0,0.0,Вологда,ТРЦ,0,58.0,58.0,0.0
85634,0,6,0,0.0,58.0,! ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) D,40,Кино - DVD,"Воронеж (Плехановская, 13)",0.0,...,0.0,0.0,0.0,0.0,Воронеж,"(Плехановская,",0,58.0,58.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42630765,33,55,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,Цифровой склад 1С-Онлайн,0.0,...,0.0,0.0,0.0,0.0,Цифровой,склад,0,4349.0,4349.0,0.0
42519299,33,56,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Чехов ТРЦ ""Карнавал""",0.0,...,0.0,0.0,0.0,0.0,Чехов,ТРЦ,0,4349.0,4349.0,0.0
42541620,33,57,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Якутск Орджоникидзе, 56",0.0,...,0.0,0.0,0.0,0.0,Якутск,"Орджоникидзе,",0,4349.0,4349.0,0.0
42564180,33,58,22169,0.0,4349.0,Яйцо дракона (Игра престолов),69,Подарки - Сувениры,"Якутск ТЦ ""Центральный""",0.0,...,0.0,0.0,0.0,0.0,Якутск,ТЦ,0,4349.0,4349.0,0.0


In [ ]:
expanded_df.isna().sum()

date_block_num               0
shop_id                      0
item_id                      0
item_cnt_day                 0
item_price                   0
item_name                    0
item_category_id             0
item_category_name           0
shop_name                    0
item_cnt_month               0
season                       0
month_sin                    0
month_cos                    0
item_cnt_month_lag_1         0
item_cnt_month_lag_2         0
item_cnt_month_lag_3         0
item_cnt_month_lag_12        0
rolling_median_6             0
rolling_median_3             0
item_avg_sales_month_lag1    0
shop_item_avg_lag1           0
log_item_cnt_month_lag_1     0
city                         0
type_of_shop                 0
price_increasing             0
item_min_price               0
item_max_price               0
price_range                  0
dtype: int64

# Save data

In [ ]:
expanded_df.to_pickle("downcasted.pkl")

In [3]:
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "projects", "sales_prediction", "datasets")

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"not found: {DATA_DIR}")

sales_train = pd.read_csv(os.path.join(DATA_DIR, "cleaned_sales.csv"))
items = pd.read_csv(os.path.join(DATA_DIR, "items.csv"))
item_categories = pd.read_csv(os.path.join(DATA_DIR, "item_categories.csv"))
shops = pd.read_csv(os.path.join(DATA_DIR, "shops.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

FileNotFoundError: not found: e:\sales_predict\notebooks\projects\sales_prediction\datasets

In [15]:
import pandas as pd

expanded_df = pd.read_parquet("../processed/features.parquet")
print(expanded_df.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../processed/features.parquet'

In [1]:
import pandas as pd

expanded_df = pd.read_parquet('../datasets/processed/features.parquet')
test = pd.read_csv('../datasets/raw/test.csv')
items = pd.read_csv('../datasets/raw/items.csv')
item_categories = pd.read_csv('../datasets/raw/item_categories.csv')
shops = pd.read_csv('../datasets/raw/shops.csv')

print(expanded_df.shape)


(9773390, 29)


# Feature Engineering for test

In [22]:
test['date_block_num'] = 34  
test_expanded = (  
    test  
    .merge(items, on='item_id', how='left')  
    .merge(item_categories, on='item_category_id', how='left')  
    .merge(shops, on='shop_id', how='left')  
)
 
def get_season_from_block_num(block_num):  
    m = (block_num % 12) + 1  
    return ("winter" if m in [12,1,2]  
            else "spring" if m in [3,4,5]  
            else "summer"  if m in [6,7,8]  
            else "autumn")

test_expanded['season']    = test_expanded['date_block_num'].map(get_season_from_block_num)  
test_expanded['month_sin'] = np.sin(2 * np.pi * test_expanded['date_block_num'] / 12)  
test_expanded['month_cos'] = np.cos(2 * np.pi * test_expanded['date_block_num'] / 12)

def month_series(m: int) -> pd.Series:
    df_m = expanded_df.loc[expanded_df['date_block_num'] == m,
                           ['shop_id','item_id','item_cnt_month']]
    return df_m.groupby(['shop_id','item_id'])['item_cnt_month'].first()

keys = list(zip(test_expanded['shop_id'], test_expanded['item_id']))

lags = {
    'item_cnt_month_lag_1': month_series(33),
    'item_cnt_month_lag_2': month_series(32),
    'item_cnt_month_lag_3': month_series(31),
    'item_cnt_month_lag_12': month_series(22)  # 34-12 = 22
}
for col, ser in lags.items():
    test_expanded[col] = ser.reindex(keys, fill_value=0).values

def rolling_median(window: int) -> np.ndarray:
    months = range(34 - window, 34)
    frames = [month_series(m) for m in months]
    df_roll = pd.concat(frames, axis=1, keys=[f"m{m}" for m in months])
    df_roll = df_roll.reindex(keys, fill_value=0)
    return df_roll.median(axis=1).values

test_expanded['rolling_median_3'] = rolling_median(3)
test_expanded['rolling_median_6'] = rolling_median(6)

test_expanded['item_avg_sales_month_lag1'] = (
    expanded_df[expanded_df['date_block_num'] == 33]
    .groupby('item_id')['item_cnt_month']
    .mean()
    .reindex(test_expanded['item_id'])
    .fillna(0)
    .values
)

test_expanded['shop_item_avg_lag1'] = (
    expanded_df[expanded_df['date_block_num'] == 33]
    .set_index(['shop_id', 'item_id'])['item_cnt_month']
    .reindex(list(zip(test_expanded['shop_id'], test_expanded['item_id'])))
    .fillna(0)
    .values
)
test_expanded['log_item_cnt_month_lag_1'] = np.log1p(test_expanded['item_cnt_month_lag_1'])

test_expanded['city']         = test_expanded['shop_name'].str.split().str[0]
test_expanded['type_of_shop'] = test_expanded['shop_name'].str.split().str[1]

item_price_lag1 = (
    expanded_df[expanded_df['date_block_num'] == 33]
    .set_index(['shop_id', 'item_id'])['item_price']
    .reindex(keys)
    .fillna(0)
    .values
)
test_expanded['item_price_lag_1'] = item_price_lag1

grp_price = expanded_df.groupby('item_id')
test_expanded['item_price']       = test_expanded['item_id'].map(grp_price['item_price'].last())
test_expanded['item_min_price']   = test_expanded['item_id'].map(grp_price['item_price'].min())
test_expanded['item_max_price']   = test_expanded['item_id'].map(grp_price['item_price'].max())

for col in ['item_price', 'item_min_price', 'item_max_price']:
    test_expanded[col] = test_expanded[col].fillna(0)

test_expanded['price_range'] = test_expanded['item_max_price'] - test_expanded['item_min_price']
test_expanded['price_range'] = test_expanded['price_range'].fillna(0)

test_expanded['price_change'] = test_expanded['item_price'] - test_expanded['item_price_lag_1']
test_expanded['price_change'] = test_expanded['price_change'].fillna(0)

test_expanded['price_increasing'] = (test_expanded['price_change'] > 0)

for c in ['item_cnt_day','item_cnt_month','date']:
    test_expanded.drop(columns=c, errors='ignore', inplace=True)

In [23]:
test_expanded

,ID,shop_id,item_id,date_block_num,item_name,item_category_id,item_category_name,shop_name,season,month_sin,...,log_item_cnt_month_lag_1,city,type_of_shop,item_price_lag_1,item_price,item_min_price,item_max_price,price_range,price_change,price_increasing
0,0,5,5037,34,"NHL 15 [PS3, русские субтитры]",19,Игры - PS3,"Вологда ТРЦ ""Мармелад""",autumn,-0.866025,...,0.000000,Вологда,ТРЦ,0.0,0.0,0.0,2599.0,2599.0,0.0,False
1,1,5,5320,34,ONE DIRECTION Made In The A.M.,55,Музыка - CD локального производства,"Вологда ТРЦ ""Мармелад""",autumn,-0.866025,...,0.000000,Вологда,ТРЦ,0.0,0.0,0.0,0.0,0.0,0.0,False
2,2,5,5233,34,"Need for Speed Rivals (Essentials) [PS3, русск...",19,Игры - PS3,"Вологда ТРЦ ""Мармелад""",autumn,-0.866025,...,0.693147,Вологда,ТРЦ,1199.0,1199.0,0.0,1199.0,1199.0,0.0,False
3,3,5,5232,34,"Need for Speed Rivals (Classics) [Xbox 360, ру...",23,Игры - XBOX 360,"Вологда ТРЦ ""Мармелад""",autumn,-0.866025,...,0.000000,Вологда,ТРЦ,0.0,0.0,0.0,1199.0,1199.0,0.0,False
4,4,5,5268,34,"Need for Speed [PS4, русская версия]",20,Игры - PS4,"Вологда ТРЦ ""Мармелад""",autumn,-0.866025,...,0.000000,Вологда,ТРЦ,0.0,0.0,0.0,0.0,0.0,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
214195,214195,45,18454,34,СБ. Союз 55,55,Музыка - CD локального производства,"Самара ТЦ ""ПаркХаус""",autumn,-0.866025,...,0.693147,Самара,ТЦ,99.0,0.0,0.0,199.0,199.0,-99.0,False
214196,214196,45,16188,34,Настольная игра Нано Кёрлинг,64,Подарки - Настольные игры,"Самара ТЦ ""ПаркХаус""",autumn,-0.866025,...,0.000000,Самара,ТЦ,0.0,1359.0,0.0,1359.0,1359.0,1359.0,True
214197,214197,45,15757,34,НОВИКОВ АЛЕКСАНДР Новая коллекция,55,Музыка - CD локального производства,"Самара ТЦ ""ПаркХаус""",autumn,-0.866025,...,0.000000,Самара,ТЦ,0.0,229.0,0.0,229.0,229.0,229.0,True
214198,214198,45,19648,34,ТЕРЕМ - ТЕРЕМОК сб.м/ф (Регион),40,Кино - DVD,"Самара ТЦ ""ПаркХаус""",autumn,-0.866025,...,0.000000,Самара,ТЦ,0.0,0.0,0.0,99.0,99.0,0.0,False


In [24]:
test_expanded.to_parquet('test.parquet')